In [ ]:
import pandas as pd
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pickle
import joblib
from sklearn.model_selection import train_test_split
import keras
import tensorflow as tf
from keras import models, layers

In [ ]:
# Connect Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define constants

# Output file path
output_file_path = '/content/drive/MyDrive/prev-experiments/cnn/data'

# Data folder path
data_folder_path = '/content/drive/MyDrive/prev-experiments/cnn/'

# Genres folder path
genres_folder_path = '/content/drive/MyDrive/prev-experiments/prev_experiments_datasets'

# CSV file header (filename + all features + genre label)
header = 'file_name chroma_stft rmse spectral_centroid spectral_bandwidth rolloff zero_crossing_rate tempo'
for i in range(1, 21):
    header += f' mfcc{i}'
header += ' label'

# Genres to analyze
genres = ['deep_house', 'tech_house', 'melodic_techno', 'progressive', 'techno_peak_time', 'hard_techno', 'minimal', 'trance']

# Batch size to train/test
batch_size = 50

train_size = 300
test_size = 100

In [ ]:
# #This code was adapted from Nicolas Gervais on https://stackoverflow.com/questions/59241216/padding-numpy-arrays-to-a-specific-size on 1/10/2021
# def padding(matrix, desired_height, desired_width):
#     current_height = matrix.shape[0]
#     current_width = matrix.shape[1]

#     top_rows_to_add = max((desired_height - current_height) // 2,0)
#     bottom_rows_to_add = max(0,desired_height - top_rows_to_add - current_height)

#     left_cols_to_add = max(0,(desired_width - current_width) // 2)
#     right_cols_to_add = max(desired_width - left_cols_to_add - current_width,0)

#     return np.pad(matrix, pad_width=((top_rows_to_add, bottom_rows_to_add), (left_cols_to_add, right_cols_to_add)), mode='constant')

In [ ]:
# max_size=1292
# n_fft=255
# hop_length = 512

# def generate_features(y, sr):
#     stft = padding(np.abs(librosa.stft(y, n_fft=n_fft, hop_length = hop_length)), 128, max_size)
#     MFCCs = padding(librosa.feature.mfcc(y, n_fft=n_fft, hop_length=hop_length,n_mfcc=128),128,max_size)
#     spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
#     chroma_stft = librosa.feature.chroma_stft(y=y, sr=sr)
#     spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
#     #Now the padding part
#     image = np.array([padding(normalize(spec_bw),1, max_size)]).reshape(1,max_size)
#     image = np.append(image,padding(normalize(spec_centroid),1, max_size), axis=0)
#     #repeat the padded spec_bw,spec_centroid and chroma stft until they are stft and MFCC-sized
#     for i in range(0,9):
#         image = np.append(image,padding(normalize(spec_bw),1, max_size), axis=0)
#         image = np.append(image, padding(normalize(spec_centroid),1, max_size), axis=0)
#         image = np.append(image, padding(normalize(chroma_stft),12, max_size), axis=0)
#     image=np.dstack((image,np.abs(stft)))
#     image=np.dstack((image,MFCCs))
#     return image

In [ ]:
# for i, genre in enumerate(genres): # Iterate over genres
#     features=[]
#     labels = []
#     for j, file_name in enumerate(os.listdir(genres_folder_path + '/' + genre + '_data')): # Iterate over each file of a specific genre
#         print('analyzing: genre ' + str(i) + '- track ' + str(j))
#         song_name = genres_folder_path + '/' + genre + '_data/' + file_name

#         # Load track using librosa
#         y, sr = librosa.load(song_name, mono=True, duration=30)

#         # Calculate features for the track
#         data = generate_features(y, sr)
#         features.append(data[np.newaxis,...])
#         labels.append(genre)
#     print('genre ' + genre + ' finished')
#     output = np.concatenate(features,axis=0)
#     try:
#       with open(output_file_path + '_' + genre + '.pickle', "wb") as f:
#             pickle.dump((np.array(output), labels), f, protocol=pickle.HIGHEST_PROTOCOL)
#     except Exception as ex:
#         print("Error during pickling object (Possibly unsupported):", ex)

In [ ]:
# for genre in genres:
#   with open(output_file_path + '_' + genre + '.pickle', "rb") as f:
#     data = pickle.load(f)
#     X_train, X_test = train_test_split(data[0], test_size=0.25, random_state=123)
#     with open(data_folder_path + genre + '_train.pickle', 'wb') as ftr:
#       pickle.dump(X_train, ftr, protocol=pickle.HIGHEST_PROTOCOL)
#     with open(data_folder_path + genre + '_test.pickle', 'wb') as fte:
#       pickle.dump(X_test, fte, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# for genre in genres:
#   with open(data_folder_path + genre + '_train.pickle', "rb") as ftr:
#     data_train = pickle.load(ftr)
#     np.save(data_folder_path + genre + '_train', data_train)
#   with open(data_folder_path + genre + '_test.pickle', "rb") as fte:
#     data_test = pickle.load(fte)
#     np.save(data_folder_path + genre + '_test', data_test)

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(genres)

LabelEncoder()

In [ ]:
import gc

def generator():
  step = 1
  while(step * batch_size <= train_size):
    for i, genre in enumerate(genres):
      with open(data_folder_path + genre + '_train.npy', "rb") as f:
        batch_start = ((step-1)*batch_size)
        batch_end = (step*batch_size)
        X_tmp = np.load(f)[batch_start:batch_end]
        del batch_start
        del batch_end
        X = np.array((X_tmp-np.min(X_tmp))/(np.max(X_tmp)-np.min(X_tmp)))
        del X_tmp
        if (i == 0):
          output = (X, label_encoder.transform(np.full(batch_size, genre)))
        else:
          output = (np.concatenate((output[0], X)), np.concatenate((output[1], label_encoder.transform(np.full(batch_size, genre)))))
        del X
        gc.collect()
    yield output
    del output
    gc.collect()
    step += 1

In [ ]:
input_shape=(128,1292,3)

cnn_model = models.Sequential()
cnn_model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
cnn_model.add(layers.MaxPooling2D((2, 2)))
cnn_model.add(layers.Flatten())
cnn_model.add(layers.Dense(32, activation='relu'))
cnn_model.add(layers.Dense(len(genres), activation='softmax'))

cnn_model.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),metrics=['accuracy'])

history = cnn_model.fit_generator(generator(), steps_per_epoch=train_size/batch_size , epochs=20, verbose=2)

Epoch 1/20


In [ ]:
# input_shape=(128,1292,3)

# cnn_model = models.Sequential()
# cnn_model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
# cnn_model.add(layers.MaxPooling2D((2, 2)))
# cnn_model.add(layers.Flatten())
# cnn_model.add(layers.Dense(32, activation='relu'))
# cnn_model.add(layers.Dense(len(genres), activation='softmax'))

# cnn_model.compile(metrics=['accuracy'])

In [ ]:
# import gc
# batch_number = 1

# for i, genre in enumerate(genres):
#   print('analyzing ' + genre)
#   with open(data_folder_path + genre + '_train.npy', "rb") as f:
#     X_tmp = np.load(f)[(batch_number-1)*batch_size:batch_number*batch_size]
#     X = np.array((X_tmp-np.min(X_tmp))/(np.max(X_tmp)-np.min(X_tmp)))
#     del X_tmp
#     if (i == 0):
#       output = (X, label_encoder.transform(np.full(batch_size, genre)))
#     else:
#       output = (np.concatenate((output[0], X)), np.concatenate((output[1], label_encoder.transform(np.full(batch_size, genre)))))
#     del X
#     gc.collect()

analyzing deep_house
analyzing tech_house
analyzing melodic_techno
analyzing progressive
analyzing techno_peak_time
analyzing hard_techno
analyzing minimal
analyzing trance


In [ ]:
# history = cnn_model.train_on_batch(x=output[0], y=output[1], reset_metrics=False)